In [27]:
!pip install opencv-python numpy pillow pandas tqdm matplotlib



[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import cv2
import numpy as np
import os
from PIL import Image

modelFile = "res10_300x300_ssd_iter_140000.caffemodel"
configFile = "deploy.prototxt.txt"

net = cv2.dnn.readNetFromCaffe(configFile, modelFile)



In [29]:
def detect_faces(img, conf=0.2):
    h, w = img.shape[:2]
    blob = cv2.dnn.blobFromImage(
        cv2.resize(img, (300,300)),
        1.0,
        (300,300),
        (104,177,123)
    )
    net.setInput(blob)
    det = net.forward()
    faces = []

    for i in range(det.shape[2]):
        if det[0,0,i,2] > conf:
            box = det[0,0,i,3:7] * np.array([w,h,w,h])
            x1,y1,x2,y2 = box.astype(int)
            x1,y1 = max(0,x1), max(0,y1)
            x2,y2 = min(w,x2), min(h,y2)
            faces.append((x1,y1,x2,y2))
    return faces

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera not accessible")
    exit()



In [ ]:
blur_mode = "gaussian"   # default
kernel_size = 31            # must be odd

frame_count = 0
faces = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Detect faces every 5 frames
    if frame_count % 5 == 0:
        facepre = cv2.resize(frame, None, fx=0.5, fy=0.5)
        faces_detect = detect_faces(facepre)
        faces = [(x1*2, y1*2, x2*2, y2*2) for x1, y1, x2, y2 in faces_detect]

    # Blur faces
    for x1, y1, x2, y2 in faces:

        # Safety check (important for video)
        if x2 <= x1 or y2 <= y1:
            continue

        face = frame[y1:y2, x1:x2]

        if blur_mode == "gaussian":
            face = cv2.GaussianBlur(face, (kernel_size, kernel_size), 0)

        elif blur_mode == "median":
            face = cv2.medianBlur(face, kernel_size)

        elif blur_mode == "average":
            face = cv2.blur(face, (kernel_size, kernel_size))

        frame[y1:y2, x1:x2] = face

    # Show blur type on screen
    cv2.putText(
        frame,
        f"Blur Mode: {blur_mode.upper()}  |  Kernel: {kernel_size}  |  G M A  + -",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    cv2.imshow("Live Camera Face Blur", frame)

    key = cv2.waitKey(1) & 0xFF

    # Switch blur modes
    if key == ord('g'):
        blur_mode = "gaussian"
    elif key == ord('a'):
        blur_mode = "average"
    elif key == ord('m'):
        blur_mode = "median"
    elif key == ord('+') or key == ord('='):
        kernel_size += 2
        if kernel_size % 2 == 0:
            kernel_size += 1
    elif key == ord('-') or key == ord('_'):
        kernel_size -= 2
        if kernel_size < 3:
            kernel_size = 3
        if kernel_size % 2 == 0:
            kernel_size -= 1
    elif key == 27:   # ESC
        break

cap.release()
cv2.destroyAllWindows()
